## FINE TUNIG BERT FOR TWITTER SENTIMENT CLASSIFICATION WITH HUGGING FACE TRANSFORMERS AND TENSORFLOW

In [ ]:
!pip install -q transformers

In [ ]:
# check for the GPU
!nvidia-smi

### Download the Data From Kaggle

The data is related to disaster tweets that can be accessed at kaggle https://www.kaggle.com/c/nlp-getting-started

### Read and Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
train_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
#test_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

In [ ]:
# check the size of the data
print("Train data size \n", train_df.shape )
print()
#print("Test data size \n", test_df.shape )

In [ ]:
# check data sample
train_df.head()

We see that for test data we do not have the 'target' columns. We will need to predict the target for the test data and submit to kaggle to check for our score.

So, for the rest of the part , we will be only working with the train data to train our model.

In [ ]:
# checking for any duplicates in the text columns
train_df['text'].duplicated().sum()

In [ ]:
# dropping the duplicates
train_df.drop_duplicates('text', keep='first')
train_df.shape

In [ ]:
# checking the target distribution
train_df.target.value_counts().plot(kind = 'bar')

We see that there is a balanaced distribution of the target levels.

In [ ]:
# checking some of the positive disaster tweets
for i in range(5):
  print(train_df[train_df.target==1]['text'].iloc[i])

In [ ]:
# checking some of the neagtive disaster tweets
for i in range(5):
  print(train_df[train_df.target==0]['text'].iloc[i])

In [ ]:
# Exploring the text lengths
textlen = train_df['text'].apply(lambda x: len(x.split()))

In [ ]:
# vsualize the text length
import seaborn as sns
plt.figure(figsize=(8,5))
plt.hist(textlen, edgecolor = "black");

We can set the maximum sequence length as 30, for training our bert model.

In [ ]:
SEQ_LEN = 30

### Fine Tuning BERT

In [ ]:
from transformers import BertTokenizer
from transformers import AutoTokenizer
import tensorflow as tf

In [ ]:
# download the tokenizer
task = "sentiment"
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

Before tokenizing the text, we can do some text cleaning to remove the user name and any http links in the tweet text.

In [ ]:
# Preprocess text (username and link placeholders)
def preprocess(text):
    new_text = []

    for t in text.split(" "):
        t = '' if t.startswith('@') and len(t) > 1 else t
        t = '' if t.startswith('http') else t
        t = t.replace("#","")
        new_text.append(t.lower())
    return " ".join(new_text).strip().replace("  ", " ")

In [ ]:
train_df['text'] = train_df['text'].apply(preprocess)
train_df.head()

**Encoding the text for Bert model**

In [ ]:
# example of tokenization
tokenizer.tokenize(preprocess("Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all"))

In [ ]:
# example of an output from encoded text by tokeneizer
encoded_input = tokenizer(preprocess("Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all"))
print(encoded_input)

In [ ]:
tokenizer.decode(encoded_input['input_ids'])

In [ ]:
# encoding the training data
train_encoded_inputs = tokenizer(train_df['text'].tolist(),
                                 add_special_tokens = True,
                                 padding='max_length', 
                                 truncation=True, 
                                 max_length=SEQ_LEN, 
                                 return_token_type_ids=False,
                                 return_tensors = 'tf')


In [ ]:
train_encoded_inputs

**Creating Tensorflow data**

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_encoded_inputs, train_df['target'].values))

In [ ]:
# formatting the data as required by bert model
def map_bert(inputs, labels):
  inputs = {'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask']}
  
  return inputs, labels

In [ ]:
train_dataset = train_dataset.map(map_bert)

In [ ]:
for t in train_dataset.take(2):
  print (t)

**Creating train and test split**

In [ ]:
dataset = train_dataset.shuffle(100000).batch(64)

In [ ]:
# length of the dataset(total batches)
DS_LEN = len(dataset)
DS_LEN

In [ ]:
# take 80% for train and 20% for validation
SPLIT = 0.8
train_ds = dataset.take(round(DS_LEN*SPLIT))
val_ds = dataset.skip(round(DS_LEN*SPLIT))

**Modeling**

In [ ]:
from transformers import TFAutoModel

In [ ]:
bert = TFAutoModel.from_pretrained('bert-base-uncased')

In [ ]:
# create model architecture
#n_classes = len((train_df.target.unique()))

# Input layers
input_ids = tf.keras.layers.Input(shape=(SEQ_LEN,), dtype=np.int32, name='input_ids' )
mask = tf.keras.layers.Input(shape=(SEQ_LEN,), dtype=np.int32, name = 'attention_mask')

# bert embeddings
embeddings = bert([input_ids, mask])[0]
cls_token = embeddings[:,0,:]

# keras layers
#x = tf.keras.layers.GlobalMaxPool1D()(embeddings)
x = tf.keras.layers.BatchNormalization()(cls_token)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.Dense(32, activation='relu')(x)

# output layer
y = tf.keras.layers.Dense(1, activation='sigmoid')(x)

# create the model
model = tf.keras.Model(inputs=[input_ids, mask], outputs=y)

In [ ]:
model.summary()

In [ ]:
# freezing the pretrained bert layer
model.layers[2].trainable = False
model.summary()

**Compile model**

In [ ]:
import transformers

In [ ]:
#learning_rate = 1e-3

optimizer = tf.keras.optimizers.Adam(learning_rate=5e-4)
#optimizer = transformers.AdamWeightDecay(learning_rate=5e-4)
loss = tf.keras.losses.BinaryCrossentropy()
metric = tf.keras.metrics.BinaryAccuracy('accuracy')

model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

In [ ]:
# train the model
callbacks = [tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.1,min_delta=0.001,monitor='val_loss'),
             tf.keras.callbacks.EarlyStopping(patience=5, min_delta=0.001, monitor='val_loss')]

history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs = 50,
    callbacks = callbacks
)

In [ ]:
# plot the performance curve
epochs = history.epoch
plt.figure(figsize=(15, 6))

# Accuracy
plt.subplot(1,2, 1)
plt.plot(epochs, history.history['accuracy'], label="Train")
plt.plot(epochs, history.history['val_accuracy'], label = "Val")
plt.legend()
plt.title("Accuracy")

# loss
plt.subplot(1,2, 2)
plt.plot(epochs, history.history['loss'], label="Train")
plt.plot(epochs, history.history['val_loss'], label = "Val")
plt.legend()
plt.title("Loss")

plt.show()

In [ ]:
# evaluate on val data
model.evaluate(val_ds)